# Deep Learning Practice 

In [ ]:
import pandas as pd
data=pd.read_csv('../Datasets/Churn_Modelling.csv')
data.head()

: 

In [ ]:
#drop the irrevelent columns
data=data.drop(
    columns=['RowNumber','CustomerId','Surname']
)
data.head()

In [ ]:
"""Encoding the Categorical data like the Gender: 
Male- 1, Female- 0, 
we use the label encoder as this has only 2 lables:
1. Male 2. female
"""
from sklearn.preprocessing import LabelEncoder
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])
data.head()

In [ ]:
"""
Since now there are 3 different cities, we use OHE to categorize 
them into a matrix. [[1,0,0],[0,0,1]],etc.
"""
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo=OneHotEncoder()
geo=onehot_encoder_geo.fit_transform(data[['Geography']])
ohe_encoder=onehot_encoder_geo.get_feature_names_out(['Geography'])

In [ ]:
#now after encoding them, we make them into an array
## We create a matrix type
geo_encoded_df=pd.DataFrame(geo.toarray(),columns=ohe_encoder)
geo_encoded_df.head()

In [ ]:
#Now we add this encoded df to the main one
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

In [ ]:
# Creating the pickle files for the encoders so that they can be used in the future
import pickle
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)
with open('onehot_encoder_geo','wb') as file:
    pickle.dump(ohe_encoder,file)

In [ ]:
#Find the independent and dependent features in the dataset
## in the dataset exited is the target variable, and remaining all are the features
X=data.drop(['Exited'],axis=1)
y=data['Exited']

In [ ]:
#Now we give the test and train split using ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=6)

In [ ]:
#scaling the data
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [ ]:
X_train

In [ ]:
#Save the Scaler data in the pickle file
with open('scaler_data.pkl','wb') as file:
    pickle.dump(scaler,file)

# ANN Implementation

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [ ]:
model=Sequential([
    Dense(70,activation='relu',input_shape=(X_train.shape[1],)),
    Dense(35,activation='relu'),
    Dense(1,activation='sigmoid')
])

In [ ]:
model.summary()

In [ ]:
#Now we give the optimizer
opt=tf.keras.optimizers.Adam(learning_rate=0.005)
loss=tf.keras.losses.BinaryCrossentropy()
loss

In [ ]:
#To train the model, we need to complie it initially
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])


In [ ]:
#declare the early stopping
early_stopping=EarlyStopping(patience=15,restore_best_weights=True,monitor='val_loss')


In [ ]:
# #Tensorboard
# # set the TensorBoard 
log_dir='logs/fit/'+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callbacks=TensorBoard(log_dir=log_dir,histogram_freq=1)


In [ ]:
#Train the model now
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=60,
    callbacks=[early_stopping,tensorboard_callbacks]
)

In [ ]:
#save the model
model.save('practice_model_1.h5')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/fit